# ⚡ 使用 GitHub 模型的并发代理工作流（Python）

## 📋 高级并行处理教程

本笔记本演示了使用 Microsoft Agent Framework 的<strong>并发工作流模式</strong>。您将学习如何构建高性能的并行处理工作流，其中多个 AI 代理同时执行，大幅提升吞吐量，并实现复杂的多线程业务流程。

## 🎯 学习目标

### 🚀 <strong>并发处理基础</strong>
- <strong>并行代理执行</strong>：同时运行多个代理，实现最大效率
- <strong>工作流编排</strong>：协调并发操作，保持数据一致性
- <strong>性能优化</strong>：通过并行处理实现显著加速
- <strong>资源管理</strong>：高效利用 AI 模型资源处理并发操作

### 🏗️ <strong>高级并发模式</strong>
- **分支-汇聚处理**：将工作拆分给多个代理并合并结果
- <strong>流水线并行</strong>：重叠执行阶段，实现连续吞吐
- <strong>负载均衡</strong>：均匀分配工作于可用代理资源
- <strong>同步点</strong>：在关键工作流阶段协调并发代理

### 🏢 <strong>企业并发应用</strong>
- <strong>高容量文档处理</strong>：同时处理多个文档
- <strong>实时内容分析</strong>：并发分析实时数据流
- <strong>批处理优化</strong>：最大化大规模操作的吞吐量
- <strong>多模态分析</strong>：并行处理不同内容类型（文本、图像、数据）

## ⚙️ 前提条件与设置

### 📦 <strong>所需依赖</strong>

安装具备并发工作流功能的 Agent Framework：

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub 模型配置**

**环境设置 (.env 文件)：**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**并发处理注意事项：**
- <strong>速率限制</strong>：监控 GitHub 模型 API 并发请求的调用限制
- <strong>资源使用</strong>：考虑多个并发代理时的内存和 CPU 使用
- <strong>错误处理</strong>：为并行操作实现健壮的错误恢复机制

### 🏗️ <strong>并发工作流架构</strong>

```mermaid
graph TD
    A[工作流开始] --> B[并发执行]
    B --> C[代理池 1]
    B --> D[代理池 2]
    B --> E[代理池 3]
    C --> F[结果汇总]
    D --> F
    E --> F
    F --> G[最终输出]
    
    H[GitHub 模型 API] --> C
    H --> D
    H --> E
```

**主要优点：**
- **⚡ 性能**：通过并行执行显著加速
- **📈 可扩展性**：处理更多工作负载且时间不成比例增加
- **🔄 效率**：更好地利用可用计算资源
- **🎯 吞吐量**：在相同时间内处理更多工作

## 🎨 <strong>并发工作流设计模式</strong>

### 🔍 <strong>研究与分析流水线</strong>
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 <strong>数据处理工作流</strong>
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 <strong>内容创作流水线</strong>
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 <strong>多阶段处理</strong>
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 <strong>企业性能优势</strong>

### ⚡ <strong>吞吐量优化</strong>
- <strong>并行执行</strong>：多个代理同时工作
- <strong>资源利用</strong>：最大化 AI 模型容量效率
- <strong>时间缩短</strong>：大幅减少总处理时间
- <strong>可扩展架构</strong>：根据需要轻松增加更多并发代理

### 🛡️ <strong>可靠性与弹性</strong>
- <strong>容错能力</strong>：单个代理故障不影响整个工作流
- <strong>错误隔离</strong>：某个并发分支的问题不影响其他分支
- <strong>优雅降级</strong>：即使代理能力减弱，系统仍持续运行
- <strong>恢复机制</strong>：自动重试和错误处理失败的操作

### 📊 <strong>监控与可观测性</strong>
- <strong>并发执行跟踪</strong>：监控所有并行操作进度
- <strong>性能指标</strong>：衡量速度提升和效率收益
- <strong>资源使用分析</strong>：优化并发代理分配
- <strong>瓶颈识别</strong>：发现并解决性能瓶颈

让我们一起构建高性能的并发 AI 工作流吧！🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
